# Churn · gradient boosting, ensembles, explanation, hand-in v2

Issue #8, second pass. Builds on `08_churn.ipynb`: same folds, same feature table, now with tuned LightGBM / XGBoost / CatBoost, two ensembles, SHAP, calibration and an error analysis.

Runtime: the Optuna tuning is 50 trials × 3 models × 5 folds, about 10–20 minutes on a laptop. The best parameters are cached in `results/best_params.json`, so the second run skips it. Delete that file to re-tune.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier

from src import style
from src.churn import (RESULTS, best_threshold, cross_validate, feature_matrix, fit_stack, rank_average,
                       save_results, summary_row, train_predict_split, tune, write_handin)
from src.data import load_customer_table
from src.features import TENURE_LABELS, build_customer_features

style.apply()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

N_TRIALS = 50
SEED = 0
FEATURE_SET = "engineered"

In [ ]:
feats = build_customer_features(load_customer_table())
train = feats[feats["split"] == "train"].reset_index(drop=True)
X = feature_matrix(train, FEATURE_SET)
y = train["churned"].astype(bool).astype(int).to_numpy()

# out-of-fold probabilities of the three baselines from 08_churn.ipynb, same folds
oof_v1 = pd.read_csv(RESULTS / "oof_predictions_v1.csv")
assert (oof_v1["customer_id"].to_numpy() == train["customer_id"].to_numpy()).all()
oof = {name: oof_v1[f"oof_{name}_{FEATURE_SET}"].to_numpy() for name in ["logreg", "random_forest", "hist_gb"]}
print(X.shape, "| baselines loaded:", list(oof))

## tuning

one search per model, objective = out-of-fold log loss on 5 folds (smoother than accuracy). the spaces are the usual ones, nothing exotic. `n_jobs=-1` everywhere.

In [ ]:
def make_lgbm(p):
    return LGBMClassifier(**p, random_state=SEED, n_jobs=-1, verbose=-1)

def space_lgbm(t):
    return dict(
        n_estimators=t.suggest_int("n_estimators", 100, 800),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=t.suggest_int("num_leaves", 8, 64),
        min_child_samples=t.suggest_int("min_child_samples", 10, 100),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10, log=True),
    )

def make_xgb(p):
    return XGBClassifier(**p, random_state=SEED, n_jobs=-1, tree_method="hist", eval_metric="logloss")

def space_xgb(t):
    return dict(
        n_estimators=t.suggest_int("n_estimators", 100, 800),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=t.suggest_int("max_depth", 2, 6),
        min_child_weight=t.suggest_int("min_child_weight", 1, 20),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10, log=True),
    )

def make_cat(p):
    return CatBoostClassifier(**p, random_seed=SEED, thread_count=-1, verbose=0, allow_writing_files=False)

def space_cat(t):
    return dict(
        iterations=t.suggest_int("iterations", 200, 1000),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        depth=t.suggest_int("depth", 3, 8),
        l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1, 30, log=True),
    )

specs = {"lightgbm": (make_lgbm, space_lgbm), "xgboost": (make_xgb, space_xgb), "catboost": (make_cat, space_cat)}
cache = RESULTS / "best_params.json"

best_params = {}
for name, (make, space) in specs.items():
    best_params[name] = tune(make, space, X, y, n_trials=N_TRIALS, seed=SEED, cache=cache, name=name)
    print(name, best_params[name])

## tuned models on the same folds as the baselines

repeated stratified 5-fold, 3 repeats, seed 0, exactly what `08_churn.ipynb` used.

In [ ]:
rows = []
for name, (make, _) in specs.items():
    p, metrics = cross_validate(make(best_params[name]), X, y, n_splits=5, n_repeats=3, seed=SEED)
    thr, acc_thr = best_threshold(y, p)
    oof[name] = p
    rows.append(summary_row(name, FEATURE_SET, metrics, thr, acc_thr))
pd.DataFrame(rows)[["model", "accuracy", "accuracy_std", "auc", "log_loss", "threshold", "accuracy_at_threshold"]]

## ensembles

- rank average of the three best models by out-of-fold AUC
- stacking: logistic regression on the out-of-fold probabilities of all six models. the base probabilities are out-of-sample, the meta-model gets its own 5-fold, so the number is only slightly optimistic

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from src.churn import oof_metrics

auc_by_model = {n: roc_auc_score(y, p) for n, p in oof.items()}
top3 = sorted(oof, key=lambda n: -auc_by_model[n])[:3]
oof["rank_avg"] = rank_average([oof[n] for n in top3])
print("rank average of", top3)

base_models = ["logreg", "random_forest", "hist_gb", "lightgbm", "xgboost", "catboost"]
oof_table = pd.DataFrame({n: oof[n] for n in base_models})
oof["stack"], m_stack = cross_validate(LogisticRegression(C=1.0, max_iter=2000), oof_table, y, n_splits=5, n_repeats=3, seed=SEED)

m_rank = {k: (v, 0.0) for k, v in oof_metrics(y, oof["rank_avg"]).items()}
for name, metrics in [("rank_avg", m_rank), ("stack", m_stack)]:
    thr, acc_thr = best_threshold(y, oof[name])
    rows.append(summary_row(name, FEATURE_SET, metrics, thr, acc_thr))

results = pd.DataFrame(rows).sort_values("accuracy_at_threshold", ascending=False).reset_index(drop=True)
results[["model", "accuracy", "accuracy_std", "auc", "log_loss", "threshold", "accuracy_at_threshold"]]

In [ ]:
# full comparison incl. the baselines from PR 1
base = pd.read_csv(RESULTS / "churn_models.csv")
base = base[(base["feature_set"] == FEATURE_SET) & base["model"].isin(["logreg", "random_forest", "hist_gb"])].drop_duplicates("model", keep="last")
cmp = pd.concat([base, results], ignore_index=True).sort_values("accuracy_at_threshold", ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(cmp["model"], cmp["accuracy_at_threshold"], color=[style.ACCENT if i == 0 else style.C[0] for i in range(len(cmp))], width=0.7)
ax.errorbar(cmp["model"], cmp["accuracy_at_threshold"], yerr=cmp["accuracy_std"], fmt="none", ecolor=style.INK2, capsize=3)
style.annotate_bars(ax, cmp["accuracy_at_threshold"], [f"{v:.3f}" for v in cmp["accuracy_at_threshold"]])
ax.set_ylim(0.72, 0.80)
style.clean(ax, "out-of-fold accuracy at the tuned threshold, 5-fold × 3", "accuracy")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
cmp[["model", "accuracy", "auc", "log_loss", "threshold", "accuracy_at_threshold"]].reset_index(drop=True)

## what the model looks at

SHAP on LightGBM fit on all labelled customers. it is the explanation model whether or not it wins the accuracy race, because tree SHAP is exact and fast and the three GBDTs agree on the ranking.

In [ ]:
import shap

explainer_model = make_lgbm(best_params["lightgbm"]).fit(X, y)
explainer = shap.TreeExplainer(explainer_model)
sv = explainer.shap_values(X)
sv = sv[1] if isinstance(sv, list) else sv

shap.summary_plot(sv, X, max_display=15, show=False, color_bar=True)
plt.gcf().set_size_inches(8, 6)
plt.title("SHAP, LightGBM: what pushes a customer towards churn", loc="left", fontweight="bold")
plt.tight_layout()

cat_savings_share and leak_share are the only features where high values push clearly towards churn (red tails at +0.3 to +0.8), that's the leakage signal on top of usage. And foreign_tx_share, cur_eur, cat_holidays all say the same: customers who use the card abroad stay. The travel card is sticky.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.flat, ["n_transactions", "days_active", "leak_share", "foreign_tx_share"]):
    xv = X[col].to_numpy()
    ax.scatter(xv, sv[:, X.columns.get_loc(col)], s=6, alpha=0.4, color=style.C[0])
    if col in ("n_transactions", "days_active"):
        ax.set_xscale("log")
    ax.axhline(0, color=style.GREY, lw=1)
    style.clean(ax, f"SHAP dependence: {col}", "SHAP value (→ churn)", col)
    ax.grid(axis="x", visible=True)
plt.tight_layout()

## calibration

the probabilities feed the value-at-risk in #10, so they should mean what they say. reliability diagram on the out-of-fold probabilities of the best single model and the stack.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color=style.GREY, lw=1, ls="--")
shown = list(dict.fromkeys([results.iloc[0]["model"], "logreg", "lightgbm"]))
for i, name in enumerate(shown):
    frac, mean_p = calibration_curve(y, oof[name], n_bins=10, strategy="quantile")
    ax.plot(mean_p, frac, marker="o", ms=5, lw=2, color=style.C[i], label=name)
style.clean(ax, "reliability, out-of-fold, 10 quantile bins", "observed churn rate", "predicted probability")
ax.grid(axis="x", visible=True)
ax.legend()
plt.tight_layout()

## where the model is wrong

out-of-fold predictions of the best model at its threshold, error rate by tenure bucket, and the confusion matrix.

In [ ]:
best_name = results.iloc[0]["model"]
thr = results.iloc[0]["threshold"]
pred = (oof[best_name] >= thr).astype(int)
err = pd.DataFrame({"tenure": train["tenure_bucket"], "wrong": pred != y, "churned": y})
by_tenure = err.groupby("tenure", observed=True).agg(error_rate=("wrong", "mean"), n=("wrong", "size"), churn_rate=("churned", "mean"))

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(by_tenure.index.astype(str), by_tenure["error_rate"] * 100, color=style.C[0], width=0.7)
style.annotate_bars(ax, by_tenure["error_rate"] * 100, [f"n={n}" for n in by_tenure["n"]])
style.clean(ax, f"{best_name}: out-of-fold error rate by tenure bucket", "% wrong")
plt.tight_layout()

cm = confusion_matrix(y, pred)
print(f"{best_name} @ {thr:.3f}  accuracy {accuracy_score(y, pred):.3f}")
print(pd.DataFrame(cm, index=["actual stay", "actual churn"], columns=["pred stay", "pred churn"]))
by_tenure.round(3)

## hand-in v2

best model by out-of-fold accuracy at its threshold. for the stack, the base models are refit on all labelled customers, the meta-model is the one fit on the out-of-fold table.

In [ ]:
X_train, y_train, X_pred, pred_ids = train_predict_split(feats, FEATURE_SET)

def fit_all_and_predict(name):
    if name in specs:
        return specs[name][0](best_params[name]).fit(X_train, y_train).predict_proba(X_pred)[:, 1]
    # baselines from PR 1, same definitions
    from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    base = {
        "logreg": make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000)),
        "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3, random_state=0, n_jobs=-1),
        "hist_gb": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=4, random_state=0),
    }[name]
    return base.fit(X_train, y_train).predict_proba(X_pred)[:, 1]

if best_name == "rank_avg":
    p_final = rank_average([fit_all_and_predict(n) for n in top3])
elif best_name == "stack":
    pred_table = pd.DataFrame({n: fit_all_and_predict(n) for n in oof_table.columns})
    p_final = fit_stack(oof_table, y).predict_proba(pred_table)[:, 1]
else:
    p_final = fit_all_and_predict(best_name)

handin = write_handin(pred_ids, p_final, thr)
print(f"{best_name} @ {thr:.3f}: predicted churn rate {handin['churned'].mean():.3f} (train rate {y.mean():.3f})")

In [ ]:
save_results(rows)
oof_df = pd.DataFrame({"customer_id": train["customer_id"], "churned": y})
for name, p in oof.items():
    oof_df[f"oof_{name}"] = np.round(p, 4)
oof_df.to_csv(RESULTS / "oof_predictions_v2.csv", index=False)
print("written results/churn_predictions.csv, churn_models.csv, oof_predictions_v2.csv")

## next (PR 3)

- TabPFN and EBM if they install cleanly
- cluster id as a feature once #7 exists
- score all 5 576 customers with the final model → `data/processed/customer_churn_scores.csv` for #10 and #12
- one paragraph in `docs/churn.md`: what predicts churn, what the model is worth, what it isn't